# Talent Target Position Distributions

This notebook compares the current talent-target distributions across positions using **Polars** for aggregation and **Bokeh** for interactive charts.

Targets included:

- `Peak_Overall`
- `Top3_Mean_Current_Overall`
- `Control_Window_Mean_Current_Overall`
- `Control_Window_Discounted_Mean_Current_Overall`


In [ ]:
from __future__ import annotations

from pathlib import Path

import polars as pl
from bokeh.io import output_notebook, show
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool, NumeralTickFormatter, TabPanel, Tabs
from bokeh.palettes import Category10
from bokeh.plotting import figure

output_notebook()

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(120)


## Load Processed Data

The notebook expects the repo root or the `notebooks/` directory as the working directory.

In [ ]:
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

features_path = repo_root / "fof8-ml" / "data" / "processed" / "features.parquet"
features_path


In [ ]:
TARGET_COLUMNS = [
    "Peak_Overall",
    "Top3_Mean_Current_Overall",
    "Control_Window_Mean_Current_Overall",
    "Control_Window_Discounted_Mean_Current_Overall",
]
BASE_COLUMNS = ["Universe", "Year", "Position_Group", *TARGET_COLUMNS]

df = (
    pl.read_parquet(features_path)
    .select([c for c in BASE_COLUMNS if c in pl.read_parquet(features_path, n_rows=1).columns])
    .with_columns(pl.col("Position_Group").cast(pl.Utf8))
    .filter(pl.col("Position_Group").is_not_null())
)

df.shape


In [ ]:
def position_summary(target_col: str, min_count: int = 250) -> pl.DataFrame:
    filtered = df.filter(pl.col(target_col).is_not_null())
    return (
        filtered.group_by("Position_Group")
        .agg(
            pl.len().alias("n"),
            pl.col(target_col).mean().alias("mean"),
            pl.col(target_col).median().alias("median"),
            pl.col(target_col).quantile(0.10).alias("p10"),
            pl.col(target_col).quantile(0.25).alias("p25"),
            pl.col(target_col).quantile(0.75).alias("p75"),
            pl.col(target_col).quantile(0.90).alias("p90"),
            pl.col(target_col).quantile(0.95).alias("p95"),
            pl.col(target_col).max().alias("max"),
        )
        .filter(pl.col("n") >= min_count)
        .sort("median", descending=True)
    )


def make_interval_plot(target_col: str, min_count: int = 250):
    summary = position_summary(target_col, min_count=min_count)
    positions = summary["Position_Group"].to_list()
    source = ColumnDataSource(summary.to_dict(as_series=False))

    fig = figure(
        x_range=positions,
        width=1200,
        height=420,
        title=f"{target_col}: position distribution intervals",
        toolbar_location="above",
    )
    fig.yaxis.axis_label = target_col
    fig.xaxis.major_label_orientation = 0.9
    fig.yaxis.formatter = NumeralTickFormatter(format="0.0")

    fig.segment("Position_Group", "p10", "Position_Group", "p90", source=source, line_width=3)
    fig.vbar(
        x="Position_Group",
        width=0.7,
        bottom="p25",
        top="p75",
        source=source,
        fill_alpha=0.45,
        line_alpha=0.9,
    )
    fig.segment("Position_Group", "median", "Position_Group", "median", source=source, line_width=10)
    fig.circle(
        x="Position_Group",
        y="mean",
        source=source,
        size=8,
        color="#d62728",
        legend_label="mean",
    )
    fig.add_tools(
        HoverTool(
            tooltips=[
                ("Position", "@Position_Group"),
                ("n", "@n{0,0}"),
                ("mean", "@mean{0.0}"),
                ("median", "@median{0.0}"),
                ("p10", "@p10{0.0}"),
                ("p25", "@p25{0.0}"),
                ("p75", "@p75{0.0}"),
                ("p90", "@p90{0.0}"),
                ("p95", "@p95{0.0}"),
                ("max", "@max{0.0}"),
            ]
        )
    )
    fig.legend.location = "top_left"
    return fig


def make_ecdf_plot(target_col: str, min_count: int = 250, top_n_positions: int = 10):
    eligible_positions = (
        df.filter(pl.col(target_col).is_not_null())
        .group_by("Position_Group")
        .agg(pl.len().alias("n"))
        .filter(pl.col("n") >= min_count)
        .sort("n", descending=True)
        .head(top_n_positions)
        .get_column("Position_Group")
        .to_list()
    )

    ecdf = (
        df.filter(pl.col("Position_Group").is_in(eligible_positions), pl.col(target_col).is_not_null())
        .select("Position_Group", pl.col(target_col).alias("target_value"))
        .sort(["Position_Group", "target_value"])
        .with_columns(
            (pl.int_range(1, pl.len() + 1).over("Position_Group")).alias("rank"),
            pl.len().over("Position_Group").alias("group_size"),
        )
        .with_columns((pl.col("rank") / pl.col("group_size")).alias("cdf"))
    )

    fig = figure(
        width=1200,
        height=420,
        title=f"{target_col}: ECDF by position (top {top_n_positions} positions by non-null count)",
        toolbar_location="above",
    )
    fig.xaxis.axis_label = target_col
    fig.yaxis.axis_label = "ECDF"
    fig.y_range.start = 0
    fig.y_range.end = 1.01

    palette = Category10[10]
    for idx, position in enumerate(eligible_positions):
        subset = ecdf.filter(pl.col("Position_Group") == position)
        source = ColumnDataSource(subset.to_dict(as_series=False))
        fig.step(
            x="target_value",
            y="cdf",
            source=source,
            mode="after",
            line_width=2,
            color=palette[idx % len(palette)],
            legend_label=position,
        )

    fig.add_tools(
        HoverTool(
            tooltips=[("value", "$x{0.0}"), ("cdf", "$y{0.00}")],
            mode="vline",
        )
    )
    fig.legend.location = "bottom_right"
    fig.legend.click_policy = "hide"
    return fig


In [ ]:
summary_tables = {
    target_col: position_summary(target_col, min_count=250)
    for target_col in TARGET_COLUMNS
}

summary_tables["Top3_Mean_Current_Overall"]


In [ ]:
panels = []
for target_col in TARGET_COLUMNS:
    interval_plot = make_interval_plot(target_col, min_count=250)
    ecdf_plot = make_ecdf_plot(target_col, min_count=250, top_n_positions=17)
    layout = column(interval_plot, ecdf_plot)
    panels.append(TabPanel(child=layout, title=target_col.replace("_", " ")))

tabs = Tabs(tabs=panels)
show(tabs)


## Notes

- The interval chart shows `p10` to `p90` as the whisker band and `p25` to `p75` as the box.
- The red marker is the mean.
- The ECDF view is useful when positions have similar medians but different upper tails.
- For the control-window targets, positions with more `null` values in recent classes will naturally have smaller eligible samples because incomplete windows are preserved rather than zero-filled.
- Positional z-scores should center each position near `0` with standard deviation near `1`, so any remaining differences come from shape, skew, truncation, or censoring rather than raw level differences.


## Positional Z-Scores

This section standardizes each talent target within `Position_Group` so the distributions are directly comparable after removing position-level mean and spread differences.


In [ ]:
Z_TARGET_COLUMNS = [f"{target_col}_Pos_Z" for target_col in TARGET_COLUMNS]

df_z = df.with_columns(
    [
        pl.when(pl.col(target_col).is_not_null())
        .then(
            pl.when(pl.col(target_col).std().over("Position_Group") > 0)
            .then(
                (pl.col(target_col) - pl.col(target_col).mean().over("Position_Group"))
                / pl.col(target_col).std().over("Position_Group")
            )
            .otherwise(0.0)
        )
        .otherwise(None)
        .alias(f"{target_col}_Pos_Z")
        for target_col in TARGET_COLUMNS
    ]
)

df_z.select(["Position_Group", *Z_TARGET_COLUMNS]).head()


In [ ]:
def position_summary_for_frame(frame: pl.DataFrame, target_col: str, min_count: int = 250) -> pl.DataFrame:
    filtered = frame.filter(pl.col(target_col).is_not_null())
    return (
        filtered.group_by("Position_Group")
        .agg(
            pl.len().alias("n"),
            pl.col(target_col).mean().alias("mean"),
            pl.col(target_col).median().alias("median"),
            pl.col(target_col).quantile(0.10).alias("p10"),
            pl.col(target_col).quantile(0.25).alias("p25"),
            pl.col(target_col).quantile(0.75).alias("p75"),
            pl.col(target_col).quantile(0.90).alias("p90"),
            pl.col(target_col).quantile(0.95).alias("p95"),
            pl.col(target_col).max().alias("max"),
        )
        .filter(pl.col("n") >= min_count)
        .sort("median", descending=True)
    )


def make_interval_plot_for_frame(frame: pl.DataFrame, target_col: str, min_count: int = 250):
    summary = position_summary_for_frame(frame, target_col, min_count=min_count)
    positions = summary["Position_Group"].to_list()
    source = ColumnDataSource(summary.to_dict(as_series=False))
    fig = figure(
        x_range=positions,
        width=1200,
        height=420,
        title=f"{target_col}: position distribution intervals",
        toolbar_location="above",
    )
    fig.yaxis.axis_label = target_col
    fig.xaxis.major_label_orientation = 0.9
    fig.yaxis.formatter = NumeralTickFormatter(format="0.00")
    fig.segment("Position_Group", "p10", "Position_Group", "p90", source=source, line_width=3)
    fig.vbar(x="Position_Group", width=0.7, bottom="p25", top="p75", source=source, fill_alpha=0.45, line_alpha=0.9)
    fig.segment("Position_Group", "median", "Position_Group", "median", source=source, line_width=10)
    fig.circle(x="Position_Group", y="mean", source=source, size=8, color="#d62728", legend_label="mean")
    fig.add_tools(HoverTool(tooltips=[("Position", "@Position_Group"), ("n", "@n{0,0}"), ("mean", "@mean{0.00}"), ("median", "@median{0.00}"), ("p10", "@p10{0.00}"), ("p25", "@p25{0.00}"), ("p75", "@p75{0.00}"), ("p90", "@p90{0.00}"), ("p95", "@p95{0.00}"), ("max", "@max{0.00}")]))
    fig.legend.location = "top_left"
    return fig


def make_ecdf_plot_for_frame(frame: pl.DataFrame, target_col: str, min_count: int = 250, top_n_positions: int = 10):
    eligible_positions = (
        frame.filter(pl.col(target_col).is_not_null())
        .group_by("Position_Group")
        .agg(pl.len().alias("n"))
        .filter(pl.col("n") >= min_count)
        .sort("n", descending=True)
        .head(top_n_positions)
        .get_column("Position_Group")
        .to_list()
    )
    ecdf = (
        frame.filter(pl.col("Position_Group").is_in(eligible_positions), pl.col(target_col).is_not_null())
        .select("Position_Group", pl.col(target_col).alias("target_value"))
        .sort(["Position_Group", "target_value"])
        .with_columns((pl.int_range(1, pl.len() + 1).over("Position_Group")).alias("rank"), pl.len().over("Position_Group").alias("group_size"))
        .with_columns((pl.col("rank") / pl.col("group_size")).alias("cdf"))
    )
    fig = figure(width=1200, height=420, title=f"{target_col}: ECDF by position (top {top_n_positions} positions by non-null count)", toolbar_location="above")
    fig.xaxis.axis_label = target_col
    fig.yaxis.axis_label = "ECDF"
    fig.y_range.start = 0
    fig.y_range.end = 1.01
    palette = Category10[10]
    for idx, position in enumerate(eligible_positions):
        subset = ecdf.filter(pl.col("Position_Group") == position)
        source = ColumnDataSource(subset.to_dict(as_series=False))
        fig.step(x="target_value", y="cdf", source=source, mode="after", line_width=2, color=palette[idx % len(palette)], legend_label=position)
    fig.add_tools(HoverTool(tooltips=[("value", "$x{0.00}"), ("cdf", "$y{0.00}")], mode="vline"))
    fig.legend.location = "bottom_right"
    fig.legend.click_policy = "hide"
    return fig


z_summary_tables = {target_col: position_summary_for_frame(df_z, target_col, min_count=250) for target_col in Z_TARGET_COLUMNS}
z_summary_tables["Top3_Mean_Current_Overall_Pos_Z"]


In [ ]:
z_panels = []
for target_col in Z_TARGET_COLUMNS:
    interval_plot = make_interval_plot_for_frame(df_z, target_col, min_count=250)
    ecdf_plot = make_ecdf_plot_for_frame(df_z, target_col, min_count=250, top_n_positions=10)
    z_panels.append(TabPanel(child=column(interval_plot, ecdf_plot), title=target_col.replace("_", " ")))

show(Tabs(tabs=z_panels))
